# Skein — Intrusion / Jamming Detector (portfolio notebook)

**Skein** is a self-healing drone-swarm mesh network that defends itself from cyber attacks and (simulated) RF jamming in real time. The *brain* of that defense is the **classic-ML detector** built here: it watches each mesh link's live network-flow features and answers **"normal, or attack?"**

This notebook is the data-science centerpiece. It runs **top-to-bottom** and covers:
1. **Data** — load real **CIC-IDS-2017** flow records, map to the shared feature schema, clean.
2. **EDA** — class balance and feature distributions (benign vs attack).
3. **Models** — compare three *classic* models: Logistic Regression, Random Forest, XGBoost. **No deep learning.**
4. **Evaluation** — confusion matrix, per-class precision/recall/F1.
5. **Feature importance** — which signals reveal an attack.
6. **Takeaways.**

The exact same `Detector` trained here is what scores live traffic in the demo, so the dashboard's predictions are genuinely data-driven — not hardcoded.

> **Data note:** this expects real CIC CSVs under `backend/data/raw/` (e.g. the cleaned [CIC-IDS-2017 V2](https://zenodo.org/records/10141593) release) **or** `$CIC_CSV` set to a CSV/dir. The pipeline never fabricates feature numbers.

In [ ]:
import os, sys

# Run from the backend/ dir so `from ml...` imports and relative data paths resolve.
NB_DIR = os.getcwd()
BACKEND = os.path.abspath(os.path.join(NB_DIR, '..', 'backend'))
if not os.path.isdir(os.path.join(BACKEND, 'ml')):
    BACKEND = os.path.abspath(os.path.join(NB_DIR, 'backend'))  # if launched from repo root
os.chdir(BACKEND)
sys.path.insert(0, BACKEND)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml.schema import FEATURE_COLUMNS, ALL_LABELS, ATTACK_TYPES, BENIGN
import ml.train as T

pd.set_option('display.width', 120)
print('backend:', BACKEND)
print('features:', FEATURE_COLUMNS)
print('labels  :', ALL_LABELS)

## 1. Load + clean the CIC data

`load_dataset()` discovers the CIC CSVs, maps their many raw column-name variants onto our 10 `FEATURE_COLUMNS`, normalizes the raw `Label` values onto `['BENIGN','DoS','PortScan','BruteForce']`, drops `inf`/`NaN` rows, and caps per class for a fast, reasonably balanced training set.

In [ ]:
df = T.load_dataset()
print('total rows:', len(df))
df.head()

## 2. EDA — class balance + feature distributions

In [ ]:
counts = df['label'].value_counts().reindex(ALL_LABELS).fillna(0).astype(int)
print(counts.to_string())

fig, ax = plt.subplots(figsize=(6,3.5))
colors = {'BENIGN':'#22c55e','DoS':'#ef4444','PortScan':'#f59e0b','BruteForce':'#3b82f6'}
ax.bar(counts.index, counts.values, color=[colors[c] for c in counts.index])
ax.set_title('class balance (after caps)'); ax.set_ylabel('rows')
plt.tight_layout(); plt.show()

In [ ]:
# Distribution of a few discriminative features, benign vs attack (log-scaled where skewed).
feats = ['flow_packets_s', 'flow_bytes_s', 'fwd_pkt_len_mean', 'flow_iat_mean']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, f in zip(axes.ravel(), feats):
    for lbl in ALL_LABELS:
        v = df.loc[df['label']==lbl, f].clip(lower=0)
        if len(v)==0:
            continue
        ax.hist(np.log1p(v), bins=50, alpha=0.5, label=lbl, color=colors[lbl])
    ax.set_title(f'log1p({f})'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Read:** attack classes separate cleanly on flow-rate and packet-length features — DoS floods show extreme `flow_packets_s`, PortScan shows many tiny forward packets. This is *why* a classic model is plenty: the signal is strong and tabular.

## 3. Train + compare three classic models

Stratified 75/25 split, `StandardScaler`, then Logistic Regression / Random Forest / XGBoost — ranked by macro-F1.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLUMNS].astype('float64'); y = df['label'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=T.RANDOM_STATE)
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)

rows = T.train_models(Xtr, Xte, y_train.values, y_test.values)
comp = pd.DataFrame([{k:r[k] for k in ('model','accuracy','precision_macro','recall_macro','f1_macro')} for r in rows])
comp = comp.sort_values('f1_macro', ascending=False).reset_index(drop=True)
comp

## 4. Evaluate the best model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best = max(rows, key=lambda r: r['f1_macro'])
model, classes = best['_obj'], list(best['_classes'])
print('best model:', best['model'], ' macro-F1 =', round(best['f1_macro'], 4), '\n')

raw_pred = model.predict(Xte)
if np.issubdtype(np.asarray(raw_pred).dtype, np.integer):
    y_pred = np.array([classes[i] for i in raw_pred])
else:
    y_pred = np.asarray(raw_pred).astype(str)

labels = [l for l in ALL_LABELS if l in set(y_test) | set(y_pred)]
print(classification_report(y_test, y_pred, labels=labels, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(cm, cmap='magma')
ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('confusion matrix')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, int(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] < cm.max()/2 else 'black', fontsize=8)
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 5. Feature importance — which signals reveal an attack

In [ ]:
if hasattr(model, 'feature_importances_'):
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS)
else:
    imp = pd.Series(np.abs(model.coef_).mean(axis=0), index=FEATURE_COLUMNS)
imp = imp.sort_values(ascending=False)
print(imp.to_string())

fig, ax = plt.subplots(figsize=(7,4))
top = imp.head(10)[::-1]
ax.barh(top.index, top.values, color='#22c55e')
ax.set_title('top feature importances'); plt.tight_layout(); plt.show()

## 6. Takeaways

- **The signal is real and strong.** On real CIC-IDS-2017 flows, the attack classes (DoS flood, PortScan recon, BruteForce) separate cleanly from benign traffic on flow-rate and packet-size features — so a *classic* tree ensemble reaches high macro-F1 without any deep learning.
- **Random Forest / XGBoost win** over Logistic Regression here because the decision boundaries are non-linear (e.g. DoS is "very high packets/sec", PortScan is "many tiny forward packets").
- **Top features are interpretable** — flow rate (`flow_packets_s`, `flow_bytes_s`), packet lengths, and inter-arrival timing dominate, which matches the physics of each attack and makes the detector explainable to a judge.
- **Deployment is honest.** The best model is saved to `models/detector.joblib`; the mesh simulator samples *real* CIC rows per link and feeds them to this exact `Detector`, so the live demo's predictions come from genuine data — never hardcoded numbers.

Run `python ml/train.py` to regenerate the artifact and `python ml/evaluate.py` for the standalone report.